# 🚀 Colab Dense Embedding - Toàn bộ Chunking (LegalRAG)

Notebook này chạy trên **Google Colab (GPU kernel)** để xây dựng **Dense Vector Index** cho **TOÀN BỘ các file chunking** trong `data/chunks/`.

- **Mô hình 1:** MiniLM-L12-v2 (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, 384 chiều)
- **Mô hình 2:** BGE-M3 (`BAAI/bge-m3`, 1024 chiều, dense)

⚡ Mỗi cell độc lập, có cơ chế dọn VRAM (`torch.cuda.empty_cache`) để tránh lỗi Out of Memory (OOM).
Mọi index được kèm file metadata `.json` ghi lại `run_id`, git commit, mô hình, số chunks, shape vector (tuân thủ quy tắc tái lập của dự án).

**Cách dùng:** Runtime → Change runtime type → T4 GPU, rồi chạy lần lượt từng cell từ trên xuống dưới.


In [ ]:
# 1. Kiểm tra GPU và môi trường PyTorch CUDA
!nvidia-smi
import torch

print("\n" + "=" * 50)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM:      {vram_gb:.2f} GB")
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> T4 GPU để tăng tốc.")
print("=" * 50)


In [ ]:
# 2. Clone mã nguồn LegalRAG và chuẩn bị môi trường Colab
import os, sys, subprocess
from pathlib import Path
from datetime import datetime, timezone

target_dir = Path("/content/LegalRAG")

if not (target_dir / "src" / "legal_rag").exists():
    print("Mã nguồn chưa tồn tại. Đang clone mới...")
    !rm -rf /content/LegalRAG
    !git clone https://github.com/ngminhduc129/LegalRAG.git /content/LegalRAG
else:
    print("Cập nhật mã nguồn mới nhất...")
    !git -C /content/LegalRAG pull

%cd /content/LegalRAG

!pip install sentence-transformers underthesea rank-bm25 typer rich loguru pydantic pydantic-yaml gitpython --quiet

src_dir = str((target_dir / "src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import legal_rag

git_commit = subprocess.check_output(["git", "-C", str(target_dir), "rev-parse", "HEAD"], text=True).strip()
run_id = f"embed_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

print(f"\n✓ LegalRAG version: {legal_rag.__version__}")
print(f"✓ Git commit: {git_commit}")
print(f"✓ Run ID: {run_id}")
print(f"✓ Working dir: {target_dir.resolve()}")


In [ ]:
# 3. Khởi tạo hàm Build Index & tự động giải phóng VRAM (Chống OOM)
import gc, json
import torch
from pathlib import Path
from legal_rag.retrieval.dense import build_dense_index

chunks_root = Path("data/chunks")
output_base_dir = Path("data/indexes/dense")
output_base_dir.mkdir(parents=True, exist_ok=True)

MODELS_CONFIG = {
    "minilm": {
        "name": "MiniLM-L12-v2",
        "model_id": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "batch_size": 32,
        "max_length": 256,
    },
    "bge_m3": {
        "name": "BGE-M3",
        "model_id": "BAAI/bge-m3",
        "batch_size": 16,  # Giữ batch_size vừa phải để tránh tràn VRAM
        "max_length": 512,
    },
}

def clear_gpu_memory():
    """Thu hồi bộ nhớ Python và xả cache VRAM GPU."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def strategy_name_for(chunk_file: Path) -> str:
    """Tạo tên strategy duy nhất từ đường dẫn tương đối so với data/chunks/."""
    rel = chunk_file.relative_to(chunks_root)
    return rel.with_suffix("").as_posix().replace("/", "__")

def run_indexing(chunk_file: Path, model_keys=["minilm", "bge_m3"], skip_existing=False):
    """Build dense index cho 1 file chunk với 1 hoặc nhiều mô hình, giải phóng VRAM ngay sau khi lưu."""
    results = []
    if not chunk_file.exists():
        print(f"❌ Không tìm thấy file chunk: {chunk_file}")
        return results

    strategy = strategy_name_for(chunk_file)

    for m_key in model_keys:
        m = MODELS_CONFIG[m_key]
        out_path = output_base_dir / f"dense_{strategy}_{m_key}.pkl"
        meta_path = output_base_dir / f"dense_{strategy}_{m_key}.meta.json"

        if skip_existing and out_path.exists():
            print(f"⏩ [Bỏ qua] Index đã tồn tại: {out_path.name}")
            continue

        print(f"\n--- [BUILD] strategy={strategy} | model={m['name']} ---")
        print(f"  Input : {chunk_file}")
        print(f"  Output: {out_path.name}")

        clear_gpu_memory()
        idx = build_dense_index(
            chunks_path=chunk_file,
            index_path=out_path,
            model_name=m["model_id"],
            batch_size=m["batch_size"],
            max_length=m["max_length"],
        )

        meta = {
            "run_id": run_id,
            "git_commit": git_commit,
            "strategy": strategy,
            "chunk_file": str(chunk_file),
            "model_key": m_key,
            "model_name": m["model_id"],
            "model_display": m["name"],
            "num_documents": len(idx.documents),
            "embedding_shape": list(idx.embeddings.shape),
            "embedding_dim": int(idx.embeddings.shape[1]),
            "created_at": datetime.now(timezone.utc).isoformat(),
        }
        meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

        print(f"✓ Hoàn thành: {meta['num_documents']} docs | Shape: {meta['embedding_shape']} | Dim: {meta['embedding_dim']}")
        print(f"  Metadata: {meta_path.name}")

        results.append(meta)

        # QUAN TRỌNG: Xóa đối tượng index và xả sạch VRAM ngay sau khi lưu file
        del idx
        clear_gpu_memory()

    return results

print("✓ Đã sẵn sàng hàm build index chống OOM!")


In [ ]:
# 4. Quét TẤT CẢ file chunk trong data/chunks/
chunk_files = sorted(chunks_root.glob("**/*.jsonl"))

print(f"Tìm thấy {len(chunk_files)} file chunk trong data/chunks/:\n")
for i, f in enumerate(chunk_files, 1):
    print(f"  {i:3d}. {f}")

# Tùy chọn: chỉ xử lý file gốc (không phải thư mục by_document/) nếu muốn chạy nhẹ hơn
PROCESS_ALL_FILES = True  # Đặt False để chỉ xử lý file gốc, bỏ qua by_document/

if PROCESS_ALL_FILES:
    files_to_process = chunk_files
else:
    files_to_process = [f for f in chunk_files if "by_document" not in f.parts]

print(f"\n➡ Sẽ xử lý {len(files_to_process)} file chunk (mỗi file × 2 mô hình = {len(files_to_process) * 2} index).")


In [ ]:
# 5. Chạy embedding TOÀN BỘ chunking (từng file × từng mô hình)
all_results = []

for chunk_file in files_to_process:
    all_results.extend(
        run_indexing(chunk_file, model_keys=["minilm", "bge_m3"], skip_existing=False)
    )

# Lưu tổng kết chạy (theo quy tắc tái lập của dự án)
summary_path = output_base_dir / "run_summary.json"
summary = {
    "run_id": run_id,
    "git_commit": git_commit,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "total_indexes": len(all_results),
    "results": all_results,
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\n{'=' * 60}")
print(f"HOÀN THÀNH: {len(all_results)} index đã tạo")
print(f"Tổng kết: {summary_path.name}")
print(f"{'=' * 60}")


In [ ]:
# 6. Kiểm tra truy vấn & Semantic Search thử nghiệm
from legal_rag.retrieval.dense import DenseIndex

test_query = "Người lao động có quyền đơn phương chấm dứt hợp đồng lao động khi nào?"

index_files = sorted(list(output_base_dir.glob("dense_*.pkl")))
print(f"Tìm thấy {len(index_files)} index để thử nghiệm.\n")
print("=" * 80)
print(f"TRUY VẤN THỬ NGHIỆM: \"{test_query}\"")
print("=" * 80)

for pkl_file in index_files:
    print(f"\n>>> Index: [{pkl_file.name}] <<<")
    idx = DenseIndex.load(pkl_file)
    results = idx.search(test_query, k=2)
    for rank, (doc_id, score, doc) in enumerate(results, 1):
        art = doc.get("article", "N/A")
        title = doc.get("article_title", "")
        clause = doc.get("clause", "")
        prefix = f"Điều {art}"
        if clause:
            prefix += f", Khoản {clause}"
        print(f"  Top {rank} (Cosine: {score:.4f}): {prefix} - {title}")
        snippet = doc.get("content", "").replace("\n", " ").strip()
        print(f"    Nội dung: {snippet[:110]}...")

    del idx
    clear_gpu_memory()


In [ ]:
# 7. Tổng kết & Nén toàn bộ Index thành ZIP để tải về
import shutil

index_files = sorted(list(output_base_dir.glob("dense_*.pkl")))
total_size_mb = sum(f.stat().st_size for f in index_files) / (1024 * 1024)

print(f"Đã tạo thành công {len(index_files)} file index (tổng cộng {total_size_mb:.2f} MB):\n")
for f in index_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  - {f.name:50} : {size_mb:.2f} MB")

zip_output = Path("/content/LegalRAG_Dense_Indexes_All")
shutil.make_archive(str(zip_output), "zip", str(output_base_dir))
zip_file = Path(f"{zip_output}.zip")
zip_size_mb = zip_file.stat().st_size / (1024 * 1024)

print(f"\n✓ Đã nén toàn bộ thành file: {zip_file.name} ({zip_size_mb:.2f} MB)")

print("\nĐể tải file ZIP về máy, chạy dòng dưới (bỏ comment):")
print("# from google.colab import files")
print("# files.download(str(zip_file))")

try:
    from google.colab import files
    # files.download(str(zip_file))  # Bỏ comment để tải tự động
    print("\n(google.colab đã sẵn sàng - bạn có thể chạy files.download(...) nếu muốn)")
except ImportError:
    pass
